In [14]:
from typing import Optional

import numpy as np
import torchgeo
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, random_split
from torchgeo.datasets import RasterDataset, stack_samples, unbind_samples, IntersectionDataset
from torchgeo.datasets.utils import download_url
from torchgeo.samplers import RandomGeoSampler, GridGeoSampler
from matplotlib.figure import Figure
from torch import Tensor
from torch.optim import Adam
from torch import nn
from torch.nn import functional as F
from collections.abc import Sequence

from backboned_unet import Unet

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 12)

In [47]:
!pwd

/home/msistlan/chi_snow


### Defining and initializing datasets

In [15]:
input_rasters_path = '../data/train/Rasters/Harmonized/'
labels_path = '../data/train/Labels/Harmonized/'
model_save_path = '../weights/model_weights_harmonized.pt'

In [16]:
class PlanetScope(RasterDataset):
    """ 
    Class for Input Dataset
    Refer: https://torchgeo.readthedocs.io/en/stable/tutorials/custom_raster_dataset.html
    """
    filename_glob = "clipped_right*.tif"
    filename_regex = "clipped_right_(?P<date>\w{10})"
    date_format = "%Y_%m_%d"
    is_image = True
    all_bands = ["Band 1", "Band 2", "Band 3", "Band 4"]
    rgb_bands = ["Band 3", "Band 2", "Band 1"]

class Lidar(RasterDataset):
    """ 
    Class for Label Dataset
    Refer: https://torchgeo.readthedocs.io/en/stable/tutorials/custom_raster_dataset.html
    """
    filename_glob = "clipped_label_right*.tif"
    filename_regex = "clipped_label_right_(?P<date>\w{10})"
    date_format = "%Y_%m_%d"
    is_image = False

<>:7: SyntaxWarning: invalid escape sequence '\w'
<>:19: SyntaxWarning: invalid escape sequence '\w'
<>:7: SyntaxWarning: invalid escape sequence '\w'
<>:19: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_705122/202699353.py:7: SyntaxWarning: invalid escape sequence '\w'
  filename_regex = "clipped_right_(?P<date>\w{10})"
/tmp/ipykernel_705122/202699353.py:19: SyntaxWarning: invalid escape sequence '\w'
  filename_regex = "clipped_label_right_(?P<date>\w{10})"


In [17]:
rgb_indices = []

for band in PlanetScope.rgb_bands:
    rgb_indices.append(PlanetScope.all_bands.index(band))

print(rgb_indices)

[2, 1, 0]


In [18]:
def preprocess_data(sample):
    """
    This function is used to do necessary preprocessing to the input image and label mask before
    passing them through the model; specifically ordering the image bands in R,G,B order and dealing with NoData pixels
    """
    image, mask = sample['image'], sample['mask']
    image = image[:, rgb_indices, :, :] # Orders RGB bands in input appropriately
    mask[mask<0] = 0
    image_empty_pos = (image <= 0).any(dim=1, keepdim=True) 
    mask[image_empty_pos] = 0 # Hack to zero out parts of label mask where image has empty data
    sample['image'] = image
    sample['mask'] = mask
    return sample

In [19]:
rasterdata = PlanetScope(input_rasters_path)

In [20]:
print(rasterdata)
print(rasterdata.res)

PlanetScope Dataset
    type: GeoDataset
    bbox: BoundingBox(minx=466068.0, maxx=472329.0, miny=3808944.0, maxy=3825267.0, mint=1485846000.0, maxt=1489129199.999999)
    size: 2
3.0


In [22]:
lidardata = Lidar(labels_path)

In [23]:
print(lidardata)
print(lidardata.res)

Lidar Dataset
    type: GeoDataset
    bbox: BoundingBox(minx=466070.22000017914, maxx=472331.22000017914, miny=3808942.0629049074, maxy=3825265.0629049074, mint=1485846000.0, maxt=1489129199.999999)
    size: 2
3.0


In [24]:
dataset = rasterdata & lidardata

In [25]:
print(dataset.res)

3.0


In [26]:
print(dataset)

IntersectionDataset Dataset
    type: IntersectionDataset
    bbox: BoundingBox(minx=466070.22000017914, maxx=472329.0, miny=3808944.0, maxy=3825265.0629049074, mint=1485846000.0, maxt=1489129199.999999)
    size: 2


In [27]:
sampler = GridGeoSampler(dataset, size=512, stride=480)
num_samples = sampler.__len__()
print(num_samples)

120


In [28]:
dataloader = DataLoader(dataset, sampler=sampler, collate_fn=stack_samples)

In [29]:
torch.cuda.get_device_name(0)

'NVIDIA A100-SXM4-80GB'

In [30]:
torch.cuda.empty_cache()

In [31]:
device = torch.device('cuda')

### Initializing UNet model and hyperparams

In [39]:
model = Unet(backbone_name='resnet50', pretrained=True, encoder_freeze=True, classes=2)

upsample_blocks[0] in: 2048   out: 256
upsample_blocks[1] in: 256   out: 128
upsample_blocks[2] in: 128   out: 64
upsample_blocks[3] in: 64   out: 32
upsample_blocks[4] in: 32   out: 16


In [40]:
model = model.to(device)

In [41]:
LEARNING_RATE = 5e-4
num_epochs = 200

In [42]:
def adjust_learning_rate_poly(optimizer, iteration):
    if iteration <= 60:
       lr = 5.0e-4
    else:
       lr = 1.0e-4

    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    return lr

In [43]:
loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

### Model Training and saving

In [45]:
model.train()
for epoch in range(num_epochs):
    print("Now in Epoch =>", str(epoch))
    adjust_learning_rate_poly(optimizer, epoch)
    epoch_loss = 0
    for item in dataloader:
        item = preprocess_data(item)
        X = item['image']
        y = item['mask']
        X = X.to(device)
        y = y.to(device)
        prediction = model(X)
        y = torch.squeeze(y,1)
        y = y.to(device)
        
        loss = loss_fn(prediction, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss = epoch_loss + loss.item()

    if epoch % 10 == 0 or epoch == num_epochs-1:
        path = f'../checkpoints/checkpoint_epoch_{epoch}.pt'
        torch.save({
            'epoch':epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': epoch_loss,
        }, path)

    print("Average loss in this epoch is "+str(epoch_loss/num_samples))

Now in Epoch => 0
Average loss in this epoch is 0.6287047723929088
Now in Epoch => 1
Average loss in this epoch is 0.586305744946003
Now in Epoch => 2
Average loss in this epoch is 0.5968705641726653
Now in Epoch => 3
Average loss in this epoch is 0.5831457669536273
Now in Epoch => 4
Average loss in this epoch is 0.5853666668136914
Now in Epoch => 5
Average loss in this epoch is 0.5402730770409108
Now in Epoch => 6
Average loss in this epoch is 0.5338741903503735
Now in Epoch => 7
Average loss in this epoch is 0.5113525453954935
Now in Epoch => 8
Average loss in this epoch is 0.5030727734168371
Now in Epoch => 9
Average loss in this epoch is 0.4883036437133948
Now in Epoch => 10
Average loss in this epoch is 0.4587146603812774
Now in Epoch => 11
Average loss in this epoch is 0.4546249931057294
Now in Epoch => 12
Average loss in this epoch is 0.423916865264376
Now in Epoch => 13
Average loss in this epoch is 0.40470733207960924
Now in Epoch => 14
Average loss in this epoch is 0.38532505

In [46]:
torch.save(model.state_dict(), model_save_path)